In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler 
from sklearn.impute import SimpleImputer 
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_breast_cancer, load_iris, load_digits
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint
from sklearn.base import BaseEstimator, TransformerMixin

# Pipeline, Column Transformer, and Custom Transformer

## Pipeline

In [3]:
data = pd.DataFrame({
    "luas": [50, 60, 70, None, 90],
    "kamar": [2, 3, 3, 4, None],
    "harga": [500, 600, 700, 800, 900]
})

# Split fitur dan target
X = data.drop("harga", axis=1)
y = data['harga']

# Train-test split 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Pipeline 
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# Training 
pipeline.fit(X_train, y_train)

# Prediksi 
prediksi = pipeline.predict(X_test)


print('Hasil prediksi:', prediksi)

Hasil prediksi: [675.]


## Column Transformer

In [5]:
data = pd.DataFrame({
    "luas": [50, 60, 70],
    "kamar": [2, 3, 4],
    "lokasi": ["A", "B", "A"],
    "harga": [500, 600, 700]
})

X = data.drop('harga', axis=1)
y = data['harga']

# Kolom 
num_cols = ['luas', 'kamar']
cat_cols = ['lokasi']

# Preprocessing 
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy="mean")),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('encoder', OneHotEncoder())
    ]), cat_cols)
])

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LinearRegression())
])

pipeline.fit(X, y)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['luas', 'kamar']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder())]),
                                                  ['lokasi'])])),
                ('model', LinearRegression())])

## Custom Transformer

In [33]:
class CustomTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns_name):
        self.column_name = columns_name

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        X_transformed[self.column_name] = X_transformed[self.column_name].apply(lambda x: x * 2)
        return X_transformed

    
data = {
    'feature_1': [1,2,3,4],
    'feature_2': [5,6,7,8]
}

df = pd.DataFrame(data)

custom_transform = CustomTransformer(columns_name='feature_1')
df_transform = custom_transform.fit_transform(df)

df_transform

,feature_1,feature_2
0,2,5
1,4,6
2,6,7
3,8,8


# Train a Model

In [7]:
data = pd.DataFrame({
    "jam_belajar": [1,2,3,4,5,6,7,8],
    "jam_tidur": [8,7,6,6,5,5,4,4],
    "lulus": [0,0,0,1,1,1,1,1]
})

X = data[["jam_belajar", "jam_tidur"]]
y = data['lulus']

# Split data (train, test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Buat model 
model = LogisticRegression()

# Training model 
model.fit(X_train, y_train)

# Testing / prediksi 
y_pred = model.predict(X_test)

# Evaluasi 
print('Accuracy:', accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



# Cross Validation

In [ ]:
# Load digits dataset 
digits = load_digits()

# Create features matrix 
features = digits.data

# Create target vector 
target = digits.target 

# Create standardizer 
standardizer = StandardScaler()

# Create logistic regression object 
logit = LogisticRegression()

# Create a pipeline that standardizes, then runs logistic regression 
pipeline = make_pipeline(standardizer, logit)

# Create k-Fold cross-validation 
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Conduct k-Fold cross-validation 
cv_results = cross_val_score(pipeline,
                             features, 
                             target,
                             cv=kf,
                             scoring='accuracy',
                             n_jobs=-1)

cv_results.mean()

0.96548417132216

In [10]:
cv_results

array([0.97777778, 0.97777778, 0.96666667, 0.95555556, 0.97222222,
       0.96666667, 0.96666667, 0.97206704, 0.96648045, 0.93296089])

### Evaluating Binary Classifier Predictions

# Hyperparameter Tuning and Hyperparameter Tuning for Feature Importances

## Hyperparamater Tuning

In [ ]:
iris = load_iris()
features = iris.data 
target = iris.target 

# Create logistic regression 
logistic = LogisticRegression()

# Create range of candidate penalty hyperparameter values 
penalty = ['l1', 'l2']

# Create range of candidates regularization hyperparameter values 
C = np.logspace(0, 4, 10)

# Create dictionary hyperparameter candidates 
hyperparameters = dict(C=C, penalty=penalty)

# Create grid search 
gridsearch = GridSearchCV(logistic, hyperparameters, cv=5, verbose=0)

# Fit grid search 
best_model = gridsearch.fit(features, target)

print('Best Penalty:', best_model.best_estimator_.get_params()['penalty'])
print('Best C:', best_model.best_estimator_.get_params()['C'])

/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Best Penalty: l2
Best C: 7.742636826811269


/usr/local/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
50 fits failed out of a total of 100.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
50 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_solve

In [ ]:
iris = load_iris()
features = iris.data 
target = iris.target 

# Create logistic regression 
logistic = LogisticRegression()

# Create range of candidate penalty hyperparameter values 
penalty = ['l1', 'l2']

# Create range of candidates regularization hyperparameter values 
C = np.logspace(0, 4, 10)

# Create dictionary hyperparameter candidates 
hyperparameters = dict(C=C, penalty=penalty)

# Create grid search 
randomizedsearch = RandomizedSearchCV(logistic, hyperparameters, cv=5, verbose=0)

# Fit grid search 
best_model = randomizedsearch.fit(features, target)

print('Best Penalty:', best_model.best_estimator_.get_params()['penalty'])
print('Best C:', best_model.best_estimator_.get_params()['C'])

/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Best Penalty: l2
Best C: 2.7825594022071245


/usr/local/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
30 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
30 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_solver

In [ ]:
# Set random seed 
np.random.seed(0)

# Load data 
iris = load_iris()
features = iris.data 
target = iris.target 

# Create a pipeline 
pipe = Pipeline([
    ('classifier', RandomForestClassifier())
])

search_space = [
    {
        "classifier": [LogisticRegression()],
        "classifier__penalty": ['l1', 'l2'],
        "classifier__C": np.logspace(0, 4, 10)},
    {
        "classifier": [RandomForestClassifier()],
        "classifier__n_estimators": [10, 100, 1000],
        "classifier__max_features": [1, 2, 3]
    }]

gridsearch = GridSearchCV(pipe, search_space, cv=5, verbose=0)

best_model = gridsearch.fit(features, target)

best_model.best_estimator_.get_params()['classifier']

/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

LogisticRegression(C=7.742636826811269)

In [ ]:
iris = load_iris()
features = iris.data 
target = iris.target 

logistic = LogisticRegression(max_iter=1000)
C = np.logspace(0, 4, 20)

hyperparameters = dict(C=C)

gridsearch = GridSearchCV(logistic, hyperparameters, cv=5, n_jobs=-1, verbose=0)

cross_val_score(gridsearch,features, target).mean()

0.9666666666666668

In [ ]:
gridsearch = GridSearchCV(logistic, hyperparameters, cv=5, verbose=1) #verbose -> mengatur seberapa banyak informasi (log/progress) yang ditampilkan saat proses berjalan

best_model = gridsearch.fit(features, target)

scores = cross_val_score(gridsearch, features, target)

print(scores.mean())

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
0.9666666666666668
